In [1]:
from pathlib import Path
import pandas as pd
from collections import defaultdict
import numpy as np
from rdkit import Chem
import pickle
from quartz.data import MoleculeData
from quartz.grids import create_grid, to_dxgrid

In [2]:
# Cluster paths
ligands_dir = Path("~/proj_FXa_QM/ligands_reduced").expanduser()
prop_dir = Path("~/proj_FXa_QM/FXa_mwfn_props/results").expanduser()
excel_path = Path("~/proj_FXa_QM/full_database_ligands.xlsx").expanduser()

In [ ]:
# Linux paths
# Load PDBs from directory and map to Excel Data
ligands_dir = Path("~/Documents/Major/Ligands/all_ligs").expanduser()
prop_dir = Path("sample_data/FXa_PDB_RESP_charges")
excel_path = Path("~/Documents/Major/Ligands/full_database_ligands.xlsx").expanduser()

In [ ]:
# Load PDBs from directory and map to Excel Data
ligands_dir = Path("~/Documents/Uni/Major/Ligands/all_ligs").expanduser()
prop_dir = Path("~/Documents/Uni/Major/CSV_props").expanduser()
excel_path = Path(
    "~/Documents/Uni/Major/Ligands/full_database_ligands.xlsx"
).expanduser()

In [ ]:
# Read Excel file
print(f"Reading Excel file: {excel_path}")
fx_data = pd.read_excel(excel_path)
# Clean column names by stripping whitespace
fx_data.columns = fx_data.columns.str.strip()

pdb_files = list(ligands_dir.glob("*_out_reduced.pdb"))

mol_data_list = []

print(f"Found {len(pdb_files)} PDB files in {ligands_dir}")

for pdb_file in pdb_files:
    # Extract PDB ID from filename (e.g. 1EZQ_out_reduced.pdb -> 1EZQ)
    pdb_id = pdb_file.name.split("_")[0]
    print(f"Processing PDB ID: {pdb_id}")

    # Path to charge file
    try:
        prop_file = list(prop_dir.glob(f"{pdb_id}*.csv"))[0]
    # print(f"Looking for charge file: {charge_file}")
    except IndexError:
        print(f"Prop file for {pdb_id} not found.")
        continue

    extra = defaultdict(float)

    with open(pdb_file, "r") as f:
        pdb_content = f.readlines()
        for line in pdb_content:
            # print(line)
            if line.startswith("REMARK"):
                # print(line.strip())
                if "minimizedAffinity" in line:
                    extra["minimizedAffinity"] = float(line.split()[-1])
                    # print("Minimized Affinity:", extra['minimizedAffinity'])
                if "CNNscore" in line:
                    extra["CNNscore"] = float(line.split()[-1])
                    # print("CNN Score:", extra['CNNscore'])
                if "CNNaffinity" in line:
                    extra["CNNaffinity"] = float(line.split()[-1])
                    # print("CNN Affinity:", extra['CNNaffinity'])

            if (
                extra["minimizedAffinity"]
                and extra["CNNscore"]
                and extra["CNNaffinity"]
            ):
                break

    # Get pKi and Ki for the PDB ID
    # Assuming column is 'PDB ID' after stripping
    row = fx_data[fx_data["PDB ID"].astype(str).str.strip() == pdb_id]

    if row.empty:
        print(f"No FX Data found for {pdb_id}")
        continue

    try:
        ki = float(row["Ki (nM)"].values[0])
    except ValueError:
        print(f"Invalid Ki value for {pdb_id}, skipping...")
        continue
    # print("Ki:", ki)
    if "pKi" in row.columns:
        pki = float(row["pKi"].values[0])
    else:
        pki = -np.log10(ki * 1e-9)  # Convert Ki in nM to pKi
    # print("pKi:", pki)

    # Store ki and pki in extra dict
    extra["ki"] = ki
    extra["pki"] = pki

    # Load Mol from PDB
    mol = Chem.MolFromPDBFile(str(pdb_file), removeHs=False)
    if mol is None:
        print(f"Failed to load PDB: {pdb_file.name}")
        continue

    prop_df = pd.read_csv(prop_file, index_col=0)  # assume the csv is a pandas export

    # Check atom count and charge assignment
    if len(prop_df) != mol.GetNumAtoms():
        print(
            f"Atom count mismatch for {pdb_id}: mol {mol.GetNumAtoms()} vs csv {len(prop_df)}"
        )
        continue

    # Assign props
    for idx, row_val in prop_df.iterrows():
        for key in prop_df.keys():
            atom = mol.GetAtomWithIdx(idx)
            atom.SetProp(key, str(float(row_val[key])))

    # Create MoleculeData (ki and pki are now in extra dict)
    md = MoleculeData(
        atomic_numbers=np.array([a.GetAtomicNum() for a in mol.GetAtoms()]),
        coordinates=mol.GetConformer().GetPositions(),
        res={},
        smiles=Chem.MolToSmiles(mol),
        rdmol=mol,
        pdb_code=pdb_id,
        filename=pdb_id,
        extra=extra,
    )
    mol_data_list.append(md)

print(f"Prepared {len(mol_data_list)} MoleculeData objects")

Reading Excel file: /scistor/gt/vie876/proj_FXa_QM/full_database_ligands.xlsx
Found 1498 PDB files in /scistor/gt/vie876/proj_FXa_QM/ligands_reduced
Processing PDB ID: 0001
Processing PDB ID: 0002
Processing PDB ID: 0003
Processing PDB ID: 0004
Processing PDB ID: 0005
Processing PDB ID: 0006
Processing PDB ID: 0007
Processing PDB ID: 0008
Processing PDB ID: 0009
Processing PDB ID: 0010
Processing PDB ID: 0011
Processing PDB ID: 0012
Processing PDB ID: 0013
Processing PDB ID: 0014
Processing PDB ID: 0015
Processing PDB ID: 0016
Processing PDB ID: 0017
Processing PDB ID: 0018
Processing PDB ID: 0019
Processing PDB ID: 0020
Processing PDB ID: 0021
Processing PDB ID: 0022
Processing PDB ID: 0023
Prop file for 0023 not found.
Processing PDB ID: 0024
Processing PDB ID: 0025
Processing PDB ID: 0026
Processing PDB ID: 0027
Processing PDB ID: 0028
Processing PDB ID: 0029
Processing PDB ID: 0030
Processing PDB ID: 0031
Processing PDB ID: 0032
Processing PDB ID: 0033
Processing PDB ID: 0034
Proce

[18:52:58] Explicit valence for atom # 14 C, 5, is greater than permitted


Processing PDB ID: 0338
Processing PDB ID: 0339
Processing PDB ID: 0340
Processing PDB ID: 0341
Processing PDB ID: 0342
Processing PDB ID: 0343
Processing PDB ID: 0344
Processing PDB ID: 0345
Processing PDB ID: 0346
Processing PDB ID: 0347
Processing PDB ID: 0348
Processing PDB ID: 0349
Processing PDB ID: 0350
Processing PDB ID: 0351
Processing PDB ID: 0352
Processing PDB ID: 0353
Processing PDB ID: 0354
Processing PDB ID: 0355
Processing PDB ID: 0356
Processing PDB ID: 0357
Processing PDB ID: 0358
Processing PDB ID: 0359
Processing PDB ID: 0360
Processing PDB ID: 0361
Processing PDB ID: 0362
Processing PDB ID: 0363
Processing PDB ID: 0364
Processing PDB ID: 0365
Processing PDB ID: 0366
Processing PDB ID: 0367
Processing PDB ID: 0368
Processing PDB ID: 0369
Processing PDB ID: 0370
Processing PDB ID: 0371
Processing PDB ID: 0372
Processing PDB ID: 0373
Processing PDB ID: 0374
Processing PDB ID: 0375
Processing PDB ID: 0376
Processing PDB ID: 0377
Processing PDB ID: 0378
Processing PDB I

[18:53:00] Explicit valence for atom # 7 C, 5, is greater than permitted


Processing PDB ID: 0464
Processing PDB ID: 0465
Processing PDB ID: 0466
Processing PDB ID: 0467
Invalid Ki value for 0467, skipping...
Processing PDB ID: 0468
Processing PDB ID: 0469
Invalid Ki value for 0469, skipping...
Processing PDB ID: 0470
Processing PDB ID: 0471
Processing PDB ID: 0472
Invalid Ki value for 0472, skipping...
Processing PDB ID: 0473
Processing PDB ID: 0474
Processing PDB ID: 0475
Processing PDB ID: 0476
Processing PDB ID: 0477
Processing PDB ID: 0478
Processing PDB ID: 0479
Processing PDB ID: 0480
Processing PDB ID: 0481
Invalid Ki value for 0481, skipping...
Processing PDB ID: 0482
Processing PDB ID: 0483
Processing PDB ID: 0484
Processing PDB ID: 0485
Processing PDB ID: 0486
Processing PDB ID: 0487
Processing PDB ID: 0488
Processing PDB ID: 0489
Processing PDB ID: 0490
Processing PDB ID: 0491
Processing PDB ID: 0492
Processing PDB ID: 0493
Processing PDB ID: 0494
Processing PDB ID: 0495
Processing PDB ID: 0496
Processing PDB ID: 0497
Processing PDB ID: 0498
Proc

[18:53:02] Explicit valence for atom # 23 O, 3, is greater than permitted


... (4 outputs left out)

Processing PDB ID: 0865
Processing PDB ID: 0866
Processing PDB ID: 0867
Processing PDB ID: 0868
Processing PDB ID: 0869
Processing PDB ID: 0870
Processing PDB ID: 0871
Processing PDB ID: 0872
Prop file for 0872 not found.
Processing PDB ID: 0873
Processing PDB ID: 0874
Processing PDB ID: 0875
Processing PDB ID: 0876
Processing PDB ID: 0877
Processing PDB ID: 0878
Processing PDB ID: 0879
Processing PDB ID: 0880
Processing PDB ID: 0881
Processing PDB ID: 0882
Processing PDB ID: 0883
Processing PDB ID: 0884
Processing PDB ID: 0885
Processing PDB ID: 0886
Processing PDB ID: 0887
Processing PDB ID: 0888
Processing PDB ID: 0889
Processing PDB ID: 0890
Processing PDB ID: 0891
Processing PDB ID: 0892
Processing PDB ID: 0893
Processing PDB ID: 0894
Processing PDB ID: 0895
Processing PDB ID: 0896
Processing PDB ID: 0897
Processing PDB ID: 0898
Processing PDB ID: 0899
Processing PDB ID: 0900
Processing PDB ID: 0901
Processing PDB ID: 0902
Processing PDB ID: 0903
Processing PDB ID: 0904
Processing

[18:53:10] Explicit valence for atom # 6 O, 4, is greater than permitted
[18:53:10] Explicit valence for atom # 6 O, 3, is greater than permitted


Processing PDB ID: 1378
Processing PDB ID: 1379
Processing PDB ID: 1380
Processing PDB ID: 1381
Processing PDB ID: 1382
Processing PDB ID: 1383
Processing PDB ID: 1384
Prop file for 1384 not found.
Processing PDB ID: 1385
Processing PDB ID: 1386
Processing PDB ID: 1387
Prop file for 1387 not found.
Processing PDB ID: 1EZQ
Processing PDB ID: 1F0R
Processing PDB ID: 1F0S
Processing PDB ID: 1FAX
Processing PDB ID: 1FJS
Processing PDB ID: 1IQE
Processing PDB ID: 1IQF
Processing PDB ID: 1IQG
Processing PDB ID: 1IQH
Processing PDB ID: 1IQI
Processing PDB ID: 1IQJ
Processing PDB ID: 1IQL
Processing PDB ID: 1IQM
Processing PDB ID: 1IQN
Processing PDB ID: 1KSN
Prop file for 1KSN not found.
Processing PDB ID: 1KXA
Processing PDB ID: 1LPG
Processing PDB ID: 1LPK
Processing PDB ID: 1LPZ
Processing PDB ID: 1LQD
Processing PDB ID: 1MQ6
Processing PDB ID: 1NFU
Processing PDB ID: 1NFW
Processing PDB ID: 1NFX
Processing PDB ID: 1NFY
Processing PDB ID: 1QB1
Processing PDB ID: 1QBN
Processing PDB ID: 1V3

[18:53:11] Explicit valence for atom # 21 C, 5, is greater than permitted


In [ ]:
# Generate grids
print("Generating grids...")
grids = create_grid(
    mol_data_list,
    props=["RESP", "polarizability"],
    size=0.4,
    threshold={"RESP": 0.001, "polarizability": 0.01},
    padding=2,
    falloff_exp=12,
)

Generating grids...
Grid lengths:  22.0 26.0 24.0
Origin:  -4.555 29.239 47.007
Cell count:  55 65 60
Processing:  0001
Processing:  0002
Processing:  0003
Processing:  0004
Processing:  0005
Processing:  0006
Processing:  0007
Processing:  0008
Processing:  0009
Processing:  0010
Processing:  0011
Processing:  0012
Processing:  0013
Processing:  0014
Processing:  0015
Processing:  0016
Processing:  0017
Processing:  0018
Processing:  0019
Processing:  0020
Processing:  0021
Processing:  0022
Processing:  0024
Processing:  0025
Processing:  0026
Processing:  0027
Processing:  0028
Processing:  0029
Processing:  0030
Processing:  0031
Processing:  0032
Processing:  0033
Processing:  0034
Processing:  0035
Processing:  0036
Processing:  0037
Processing:  0038
Processing:  0039
Processing:  0040
Processing:  0041
Processing:  0042
Processing:  0043
Processing:  0044
Processing:  0045
Processing:  0046
Processing:  0047
Processing:  0048
Processing:  0049
Processing:  0050
Processing:  005

In [7]:
# Save to pkl
output_file = "grids_12.pkl"
with open(output_file, "wb") as f:
    pickle.dump(grids, f)

In [ ]:
# Save to DX
# print("Saving to DX format...")
# to_dxgrid(grids)